In [ ]:
from pyspark.sql.session import SparkSession
spark = (SparkSession.builder
                .master("local")
                .appName("withColumnFunctionUsage")
                .getOrCreate())

help(SparkSession)

: 

In [ ]:
people = spark.createDataFrame([
            {"deptId": 1, "age": 40, "name": "Hyukjin Kwon", "gender": "M", "salary": 50},
            {"deptId": 1, "age": 50, "name": "Takuya Ueshin", "gender": "M", "salary": 100},
            {"deptId": 2, "age": 60, "name": "Xinrong Meng", "gender": "F", "salary": 150},
            {"deptId": 3, "age": 20, "name": "Haejoon Lee", "gender": "M", "salary": 200}
         ])

age_col = people.age
age_col

In [ ]:
department = spark.createDataFrame([
            {"id": 1, "name": "PySpark"},
            {"id": 2, "name": "ML"},
            {"id": 3, "name": "Spark SQL"}
         ])

people.filter(people.age > 30).show()

people.where("age > 30").show()

# df.filter() and df.where() are essentially the same and can be used interchangeably. The choice between them is mostly a matter of style and readability. Some developers prefer filter() for its straightforwardness, while others prefer where() for its SQL-like syntax. In practice, you can use either one based on your preference or the conventions of your codebase.
# There is no performance difference between them as they generate the same execution plan.
# You can use column expressinos or SQL style strings -- both are supported by filter() and where().
# Ex: using column expression:
df.filter(df.age > 30).show()  # Using column expression
df.where(df.city == "New York").show()  # Using SQL style string

# Ex: using SQL style string:
df.filter("age > 30").show()
df.where("city = 'New York'").show()

# chaining multiple conditions
people.filter((people.age > 30) & (people.gender == "M")).show()  # Using column expressions


In [ ]:
people.filter(people.age > 30).join(
    department, people.deptId == department.id
)\
.groupBy(department.name, people.gender)\
.agg({"salary": "avg", "age": "max"}).show()

In [ ]:
department.select("name").show()
department.select(department.name).show()
department.select(department["name"]).show()

In [ ]:
df = spark.range(3)
# dir(df) lists all the attributes(columns & methods) available on the supplied object
# here it lists all the methods that are available on the DataFrame object
dir(df)

In [ ]:
# lists all the attributes (columns and methods) that starts with 'i' and present in DataFrame
[attr for attr in dir(df) if attr[0] == 'i']

In [ ]:
from pyspark.sql.functions import *
# # add a new column named 'i_like_pancakes'
# it's not going to alter your existing df, instead returns a new df with the new column
df = df.withColumn('i_like_pancakes', lit(1))
[attr for attr in dir(df) if attr[0] == 'i']

In [ ]:
# try to add an existed column 'inputFiles' -- no change/overwrites the existing column
df = df.withColumn('inputFiles', lit(3))
[attr for attr in dir(df) if attr[0] == 'i']

In [ ]:
# Dont include columns that are not valid python identifiers
df.withColumn('1', lit(4)).withColumn('name 1', lit(5)).show()

In [ ]:
df = spark.createDataFrame([(2, "Alice"), (5, "Bob")], schema=["age", "name"])
df.show()
print(type(df.age))
print(type(df.select(df.age)))
df.select(df.age).show()

In [ ]:
# select columns based on indexes
df.select(df[0]).show()
df.select(df[1]).show()

In [ ]:
# select multiple string columns as index
df.select("name", "age").show()
df[df.age > 3].show()

In [ ]:
# agg --  Aggregate on the entire :class:`DataFrame` without groups
df = spark.createDataFrame([(2, "Alice"), (5, "Bob")], schema=["age", "name"])

# duplicate keys in dict overwrites the older value. Hence if you need multiple aggregation on the same column use 
df.agg({"age": "max", "age": "avg"}).show()

from pyspark.sql.functions import max, avg, min
df.agg(max("age"), avg("age"), min("age")).show()

In [ ]:
# alias(self, alias: str) -> 'DataFrame'
# Returns a new :class:`DataFrame` with an alias set.

df = spark.createDataFrame([(14, "Tom"), (23, "Alice"), (16, "Bob")], ["age", "name"])

df_as1 = df.alias("df_as1")
df_as2 = df.alias("df_as2")

df_as1.show()
df_as2.show()

# by default -- it does inner join
joined_df = df_as1.join(df_as2, df_as1.age == df_as2.age)
joined_df.show()

# sort(desc(col)) sorts the df in desc order on basis of specified column
joined_df.select("df_as2.age", "df_as1.name").sort(desc("df_as2.age")).show()

In [ ]:
# cache(self) -> 'DataFrame'
#  |      Persists the :class:`DataFrame` with the default storage level (`MEMORY_AND_DISK_DESER`).
df = spark.range(3)
df.cache()

In [ ]:
# Prints the (logical and physical) plans to the console for debugging purposes.
# Parameters
#     ----------
#     extended : bool, optional
#         default ``False``. If ``False``, prints only the physical plan.
#         When this is a string without specifying the ``mode``, it works as the mode is
#         specified.
#     mode : str, optional
#         specifies the expected output format of plans.
    
#         * ``simple``: Print only a physical plan.
#         * ``extended``: Print both logical and physical plans.
#         * ``codegen``: Print a physical plan and generated codes if they are available.
#         * ``cost``: Print a logical plan and statistics if they are available.
#         * ``formatted``: Split explain output into two sections: a physical plan outline                 and node details.


# by default -- only prints physical plans

df.explain()

# what are phyical and logical plans in spark?
#     When you write Spark code, Spark does not execute it immediately.
#     Instead, Spark goes through multiple planning stages before actually running anything:
        
#         Your Spark code
#             ↓
#         Logical Plan
#             ↓ (optimized by Catalyst)
#         Optimized Logical Plan
#             ↓
#         Physical Plan
#             ↓
#         Execution on cluster

# Logical Plan (WHAT to do)
#   A Logical Plan describes what computation needs to be done, not how it will be executed.
#   It is:
#     Abstract
#     Declarative
#     Independent of cluster resources
    
# 📌 Example
# df.filter(df.age > 30).select("name")

# Logical plan might look like:

# Project [name]
#  └─ Filter (age > 30)
#      └─ Scan people table

# This means:
# Read data
# Filter rows where age > 30
# Select name column

# 👉 No information about joins, shuffles, partitions, or algorithms yet.

# 3️⃣ Optimized Logical Plan
# Before execution, Spark’s Catalyst Optimizer improves the logical plan by applying rules like:
# Predicate pushdown
# Column pruning
# Constant folding
# Reordering filters

# Example optimization
# df.select("name", "age").filter(df.age > 30)

# Optimized to:

# Filter (age > 30)
#  └─ Project [name, age]


# 4️⃣ Physical Plan (HOW to do it)
# 📌 Definition

# A Physical Plan defines how Spark will actually execute the job.

# It includes:
# Execution strategies
# Algorithms
# Shuffles
# Partitioning
# Operators
    
# Example Physical Plan
# *(1) Project [name]
#  └─ *(1) Filter (age > 30)
#      └─ *(1) FileScan parquet people

# Now Spark knows:
# Read Parquet files
# Apply filter during scan
# Use specific operators
# How many stages/tasks to create

# 5️⃣ Key differences (very important)
# Aspect	    Logical Plan	Physical Plan
# Focus	    What to compute	How to compute
# Level	    High-level	    Low-level
# Optimization	Rule-based	Cost-based
# Cluster details	❌ No	    ✅ Yes
# Shuffles/joins	❌ No	    ✅ Yes
# Execution ready	❌ No	    ✅ Yes

# 7️⃣ Why this matters in real projects

# Understanding plans helps you:

# Debug slow jobs
# Identify unnecessary shuffles
# Verify predicate pushdown
# Optimize joins and filters
# Write efficient transformations

# Example:
# Filter not pushed down → slow scan
# Wrong join type → large shuffle

# Summary:
# In Spark, a logical plan defines what operations need to be performed, while a physical plan defines how those operations will be executed on the cluster. 
# Spark uses the Catalyst optimizer to convert the logical plan into an optimized logical plan and then generates one or more physical plans, choosing 
# the most efficient one for execution.

In [ ]:
# `extended``: Print both logical and physical plans.
df.explain('extended')

In [ ]:
# checkpoint(self, eager: bool = True) -> 'DataFrame'
#  |      Returns a checkpointed version of this :class:`DataFrame`. Checkpointing can be used to
#  |      truncate the logical plan of this :class:`DataFrame`, which is especially useful in
#  |      iterative algorithms where the plan may grow exponentially. It will be saved to files
#  |      inside the checkpoint directory set with :meth:`SparkContext.setCheckpointDir`.
 
    
#      Parameters
#  |      ----------
#  |      eager : bool, optional, default True
#  |          Whether to checkpoint this :class:`DataFrame` immediately.



import tempfile
df = spark.createDataFrame([
            (14, "Tom"), (23, "Alice"), (16, "Bob")], ["age", "name"])

from tempfile import TemporaryDirectory() as d:
    # all the logical plans will be stored here
spark.sparkContext.setCheckpointDir("/tmp/bb")
df.checkpoint(False)
    
# 1️⃣ What is checkpointing in Spark?

# Checkpointing is a mechanism where Spark:
# 1. Materializes a DataFrame/RDD
# 2. Writes it to reliable storage (HDFS / S3 / local FS for dev)
# 3. Cuts off (truncates) the logical lineage
# After checkpointing, Spark stops recomputing from the original transformations and instead reads from the checkpointed data.
# This is especially important in long, complex, or iterative pipelines.
    
# 2️⃣ Why checkpointing exists (the real problem it solves)
# 🔹 Problem: Lineage explosion

# Spark is lazy. Every transformation adds to a logical plan (lineage DAG).In scenarios like:
# 1. Iterative algorithms
# 2. Repeated joins/aggregations
# 3. Streaming/stateful pipelines

# …the lineage can become:
# 1. Very large
# 2. Expensive to recompute on failure
# 3. Hard for the optimizer to manage

# 🔹 Solution: Checkpoint
# Checkpointing:
# 1. Breaks the lineage
# 2. Creates a new starting point
# 3. Improves fault tolerance and stability

# 3️⃣ What exactly happens internally?
# When you call:
# df.checkpoint()

# Spark does the following:
# 1. Executes the DataFrame (action is triggered)
# 2. Writes the output to the checkpoint directory
# 3. Replaces the original logical plan with a FileScan
# 4. Future operations read from checkpointed files.

# 👉 Logical plan before:
# Scan → Filter → Join → Aggregate → ...

# 👉 Logical plan after checkpoint:
# FileScan (checkpoint data) → next transformations

# This is why the docstring says:
# “truncate the logical plan”.

# 4️⃣ Checkpoint vs Cache (very important distinction)
# Aspect	        Cache/Persist	    Checkpoint
# Storage	        Memory / Disk	    Reliable storage
# Lineage truncatedx ❌ No	            ✅ Yes
# Fault tolerance	    ❌ Limited	    ✅ Strong
# Recomputed on failure ✅ Yes	        ❌ No
# Cost	            Cheap	        Expensive

# 👉 Rule of thumb

# Cache = performance optimization

# Checkpoint = Fault tolerance & stability optimization.

# 5️⃣ Eager vs Non-Eager checkpointing
# Method signature
# checkpoint(eager: bool = True)

# 🔹 eager=True (default)

# 1. Checkpoint happens immediately
# 2. Spark triggers a job right away
# 3. Safer, deterministic.

# df_cp = df.checkpoint()

# 🔹 eager=False

# 1. Checkpointing is lazy
# 2. Happens only when an action is called later.

# df_cp = df.checkpoint(False)
# df_cp.show()  # checkpoint happens here

# df.checkpoint(False)
# 1. Marks the DataFrame for checkpointing
# 2. Does not execute immediately
# 3. Lineage will be cut once an action is triggered

# 👉 Use eager=False when:

# 1. You want Spark to optimize execution further
# 2. You don’t want immediate job submission.

# Why setCheckpointDir() is mandatory
# spark.sparkContext.setCheckpointDir("/tmp/bb")

# Checkpoint data must be stored in a fault-tolerant location.
# In prod, it could be HDFS, S3 or ADLS. In dev, local FS is fine but must be set explicitly.
# If you don’t set this → Spark throws an error.

# spark.sparkContext.setCheckpointDir("/tmp/bb")
# Defines where Spark will persist checkpoint data.


# 8️⃣ When should you use checkpointing?
# ✅ Good use cases

# 1. Iterative ML algorithms
# 2. Graph processing
# 3. Complex dbt-style transformation chains
# 4. Streaming with state
# 5. Very deep transformation DAGs

# ❌ Avoid when
# 1. Small datasets
# 2. Simple pipelines
# 3. One-time transformations

# Checkpointing is expensive (I/O heavy).

# SUMMARY
# Checkpointing in Spark is used to truncate the logical lineage of a DataFrame by materializing it to reliable storage. 
# This helps prevent lineage explosion, improves fault tolerance, and stabilizes long or iterative pipelines. 
# Unlike caching, checkpointing breaks the dependency on previous transformations and allows Spark to recover without recomputation.

In [ ]:
# coalesce(numPartitions) --> DataFrame
# Coalesce is mainly used to reduce the num of partitions. 
# Avoids shuffling (or minimises it)
# Faster, but can create skewed partitions

# repartition(numPartition, columns)
# Repartition can increase or decrease partitions
# Uses full shuffle of data
# Slower, but produces evenly balanced partitions

# When to use what

# Use repartition when:
#     You need more partitions
#     You need balanced data
#     You repartition by a column

# Use coalesce when:
#     You just want to reduce partitions quickly
#     Data skew is not a big concern
#     Before writing small number of output files


df = spark.range(10)

# reduce the num of partitions to 1
df.coalesce(1).rdd.getNumPartitions()


# coalesce() and repartition() solve one of Spark’s core problems: how data is physically distributed across partitions.

# 1️⃣ Why partitions matter in Spark (core concept)

# Spark processes data in parallel using partitions.
# Partitions affect:
# 1. Parallelism (CPU usage)
# 2. Shuffle cost
# 3. Job runtime

# File sizes written to storage

# 👉 Too many partitions → overhead
# 👉 Too few partitions → underutilized cluster

# coalesce() and repartition() exist to control this.


# 2️⃣ What is repartition()?
# repartition(n) changes the number of partitions by doing a full shuffle.

# df2 = df.repartition(200)

# 🔹 What happens internally?
# 1. Spark redistributes all rows across the cluster
# 2. Uses a shuffle
# 3. Ensures even distribution

# 🔹 When to use
# 1. Before joins (to avoid skew)
# 2. Before heavy aggregations
# 3. When increasing parallelism
# 4. When data is badly skewed


# 3️⃣ What is coalesce()?
# coalesce(n) reduces the number of partitions without a shuffle (by default).

# df2 = df.coalesce(10)

# 🔹 What happens internally?
# 1. Combines existing partitions
# 2. Moves data minimally
# 3. Avoids shuffle

# 🔹 When to use
# 1. Before writing to storage
# 2. Reducing small files
# 3. Post-filtering where data volume shrinks
# 4. Performance optimization

# 4️⃣ Coalesce with shuffle (important nuance)
# df.coalesce(10, shuffle=True)

# This behaves like:
# 1. A cheaper repartition
# 2. Allows better balance
# 3. Still less flexible than repartition()

# 5️⃣ Side-by-side comparison (interview gold)

# Feature	                    repartition()	coalesce()

# Shuffle	                    Always	        No (default)
# Can increase partitions	    Yes	            No
# Can decrease partitions	    Yes	            Yes
# Data evenly balanced	        Yes	            Not guaranteed
# Performance cost	            High	        Low
# Common use case	            Before joins    Before writes

# 6️⃣ Real-world examples
# Example 1: Fix small files before write
# df.coalesce(10).write.parquet("s3://bucket/output/")

# ✔ Reduces small files
# ✔ Avoids shuffle

# Example 2: Improve join performance
# df.repartition(200, "customer_id")

# ✔ Better parallelism
# ✔ Reduced skew


# 7️⃣ How this ties to logical & physical plans

# 1. Logical plan: unchanged (Spark doesn’t care)
# 2. Physical plan: partitioning strategy changes
# 3. repartition() introduces a ShuffleExchange
# 4. coalesce() usually does not

# 8️⃣ Common mistakes 🚨

# ❌ Using repartition() before every write
# ❌ Using coalesce() to increase partitions
# ❌ Ignoring data skew
# ❌ Hardcoding partition counts

# 9️⃣ Interview-ready summary 🎯

# repartition() and coalesce() control how data is distributed across partitions in Spark.
# repartition() performs a full shuffle and is used to increase or rebalance partitions, while coalesce() reduces partitions with minimal data movement 
# and is typically used to optimize writes and reduce small files.

In [ ]:
#   collect(self) -> List[pyspark.sql.types.Row]
#  |      Returns all the records as a list of :class:`Row`.

df = spark.createDataFrame(
        [(14, "Tom"), (23, "Alice"), (16, "Bob")], ["age", "name"])
df.collect()

In [ ]:
# corr(col1, col2) -- Calculates the correlation of two columns of a :class:`DataFrame` as a double value.

# corr() in PySpark is used to measure how strongly two numeric columns are related to each other.

# What it actually does
# corr(col1, col2) computes the Pearson correlation coefficient between two columns.
# Value range: -1 to +1

# Meaning:
# +1 → perfect positive relationship
# -1 → perfect negative relationship
# 0 → no linear relationship

# It helps you answer questions like:
# Do two metrics move together?
#     Example: ad_spend vs revenue
# Is one variable a good predictor of another?
# Is there redundancy between features in ML?
# Should two columns be combined, removed, or transformed?

df = spark.createDataFrame([(1, 12), (10, 1), (19, 8)], ["c1", "c2"])
df.corr("c1", "c2")

In [ ]:
# count(self) -> int
#  |      Returns the number of rows in this :class:`DataFrame`.
df = spark.createDataFrame(
        [(14, "Tom"), (23, "Alice"), (16, "Bob")], ["age", "name"])
df.count()

In [ ]:
# createGlobalTempView(self, name: str) -> None
#  |      Creates a global temporary view with this :class:`DataFrame`.
#  |      
#  |      The lifetime of this temporary view is tied to this Spark application.
#  |      throws :class:`TempTableAlreadyExistsException`, if the view name already exists in the
#  |      catalog.

df = spark.createDataFrame([(2, "Alice"), (5, "Bob")], schema=["age", "name"])
df.createGlobalTempView("people")
df2 = spark.sql("select * from global_temp.people")

In [ ]:
# very good way to compare if 2 Dataframes are equivalent
# call df.collect() -- this returns list of Row() objects. Sort the list and compare
sorted(df2.collect()) == sorted(df.collect())

In [ ]:
# drop the GlobalTempView.
spark.catalog.dropGlobalTempView("people")

In [ ]:
# createOrReplaceGlobalTempView(self, name: str) -> None
#  |      Creates or replaces a global temporary view using the given name.
#  |      
#  |      The lifetime of this temporary view is tied to this Spark application.

df = spark.createDataFrame([(2, "Alice"), (5, "Bob")], schema=["age", "name"])
df.createGlobalTempView("people")

In [ ]:
# replace the global temp view 
df2 = df[df.age > 3]    # Pandas like syntax to filter the DataFrame, but less explicit. We prefer filter() or where() for better readability and explicitness. 
# df[df.age > 3] is essentially same as df.filter(df.age > 3) or df.where(df.age > 3).
# If you prefer SQL style string, you can also do df.where("age > 3") or df.filter("age > 3") to achieve the same result.
# df[df.age > 3] is just a syntactic sugar for filter()
# Important Distinction: df[condidtion] -> filter whereas df["column_name"] -> column
# No functional difference. Generates same logical and physical plans as filter() or where(). Same performance.

# 6️⃣ Interview-ready answer 🎯
# Yes, df[df.condition], df.filter(), and df.where() all perform row filtering in PySpark. They are functionally equivalent and generate the 
# same execution plan. The bracket syntax is just syntactic sugar, while filter() and where() are more explicit and commonly used in production code.
 
df2.createOrReplaceGlobalTempView("people")

In [ ]:
df3 = spark.sql("select * from global_temp.people")
sorted(df2.collect()) == sorted(df3.collect())

In [ ]:
spark.catalog.dropGlobalTempView("people")

In [ ]:
#  createOrReplaceTempView(self, name: str) -> None
#  |      Creates or replaces a local temporary view with this :class:`DataFrame`.
#  |      
#  |      The lifetime of this temporary table is tied to the :class:`SparkSession`
#  |      that was used to create this :class:`DataFrame`.


# 👉 createOrReplaceTempView() creates a view, not a table.
# It creates a temporary view:
# 1. ✅ Logical object (metadata only)
# 2. ❌ No data is physically stored
# 3. ❌ Not a table
# 4. ⏳ Exists only for the lifetime of the SparkSession

df = spark.createDataFrame([(2, "Alice"), (5, "Bob")], schema=["age", "name"])
df.createOrReplaceTempView("people")

# After this, Spark registers people as a logical view over df.

# 2️⃣ How it behaves (key characteristics)
# Aspect	                     Temp View
# Object type	                 View
# Data stored	                ❌ No
# Backed by storage	        ❌ No
# Lifetime	                 SparkSession
# Accessible via SQL	        ✅ Yes
# Persistent across sessions	❌ No

# 3️⃣ What happens under the hood?
# Spark stores:
#     The logical plan of df
#     A mapping: people → df logical plan

# When you run:

# SELECT * FROM people


# Spark:

# 1. Resolves people to the DataFrame’s logical plan
# 2. Optimizes it
# 3. Executes it lazily

# 👉 No data is materialized unless an action is triggered.


# 5️⃣ Local temp view vs Global temp view (important)
# Local temp view (your example)
# df.createOrReplaceTempView("people")

# Visible only in this SparkSession

# Global temp view
# df.createOrReplaceGlobalTempView("people")

# Visible across sessions

# Accessed as:
# SELECT * FROM global_temp.people

# Still a view, not a table.


# 7️⃣ Interview-ready answer 🎯

# createOrReplaceTempView() creates a temporary view, not a table. It does not store data on disk; 
# it only registers the DataFrame’s logical plan under a name so it can be queried using Spark SQL. 
# The view exists only for the lifetime of the SparkSession.

In [ ]:
# How can we create a table out of a dataframe?

# You can create a table out of a DataFrame in Spark by materializing the DataFrame to storage. 
# There are a few common ways to do this, depending on whether you want the table to be temporary or persistent.

# 1️⃣ Create a managed table (most common)

# A managed table is fully owned by Spark/Hive. Spark decides where data is stored.

# df.write.saveAsTable("people")

# What this does
# 1. Writes data to disk
# 2. Registers a table in the metastore
# 3. Table persists across Spark sessions

# You can now query it using SQL:
# SELECT * FROM people;


# 2️⃣ Create a table using SQL (CTAS)

# You can also use Create Table As Select:

# df.createOrReplaceTempView("people_tmp")

# spark.sql("""
# CREATE TABLE people
# AS SELECT * FROM people_tmp
# """)

# This is equivalent to saveAsTable() but SQL-driven.


# 3️⃣ Create an external table

# If you want Spark to not own the data location:

# df.write \
#   .format("parquet") \
#   .option("path", "/data/people") \
#   .saveAsTable("people")

# Key difference
# 1. Dropping the table does not delete the data
# 2. Useful when data is shared across systems


# 4️⃣ Append vs overwrite when creating tables
# Overwrite (replace table data)

# df.write.mode("overwrite").saveAsTable("people")

# Append
# df.write.mode("append").saveAsTable("people")

# ⚠️ Append requires schema compatibility.

# 6️⃣ Comparison summary
# Method	                Creates Table?	Persistent?	Data Stored?
# createOrReplaceTempView	❌ View	        ❌	        ❌
# saveAsTable	            ✅	            ✅	        ✅
# CTAS	                    ✅	            ✅	        ✅
# External table	        ✅	            ✅	        ✅

# 7️⃣ Interview-ready summary 🎯

# To create a table from a DataFrame in Spark, we use df.write.saveAsTable() or a CTAS query. 
# This materializes the DataFrame to storage and registers it in the metastore. 
# Unlike temp views, tables persist across Spark sessions and store data physically.

In [ ]:
# so does saveAsTable() saves the data on my local machine? and where is the metastore in this case?

# 👉 Yes, if you are running Spark in local mode (master("local[*]")).

# In your setup:

# SparkSession.builder.master("local[*]").getOrCreate()

# Spark is running on your local machine
# There is no distributed filesystem
# So Spark writes data to your local disk

# Where exactly on local disk?

# By default, Spark writes managed tables to:

# spark.sql.warehouse.dir

# Example:
# ~/spark-warehouse/people/

# You can check it using:
    
# spark.conf.get("spark.sql.warehouse.dir")


# 2️⃣ What is the metastore in this case?

# When running locally, Spark uses an embedded Hive metastore.

# Key points:
# 1. Metastore is backed by Apache Derby
# 2. Stored locally (not a service)
# 3. Used only by your SparkSession

# Typical location:
# metastore_db/

# This directory contains:
# 1. Table metadata
# 2. Schema
# 3. Table location
# 4. Table type (managed/external)

# 👉 So in local mode:
# Data → local filesystem (spark-warehouse)
# Metadata → local Derby metastore


# 3️⃣ What happens when you restart Spark?

# If you restart Spark from the same directory:

# Spark can still see the tables

# If you:
# 1. Delete metastore_db
# 2. Change working directory

# 👉 Tables will appear missing, even though files may still exist.

# This is why local mode is not production-grade.


# 4️⃣ How this changes in a real cluster (important)
# Environment	        Data location	        Metastore
# Local mode	        Local disk	            Embedded Derby
# Cluster (prod)	    HDFS / S3 / ADLS	    Hive Metastore service
# Databricks	        DBFS / Cloud storage	Managed metastore

# In production:
# 1. Data is not on driver machine
# 2. Metastore is centralized
# 3. Tables are visible across sessions & users


# 5️⃣ Managed vs external table (local mode)
# df.write.saveAsTable("people")
# 1. Managed table
# 2. Spark controls data lifecycle
# 3. Dropping table deletes files


# df.write.option("path", "/tmp/people").saveAsTable("people")
# 1. External table
# 2. Dropping table does NOT delete files


# 7️⃣ Interview-ready summary 🎯

# When running Spark in local mode, saveAsTable() writes data to the local filesystem under the Spark warehouse directory
# and registers metadata in an embedded Derby-based Hive metastore. 
# In production clusters, data is written to distributed storage and metadata is stored in a centralized Hive metastore service.

In [ ]:
# replace the local temp view 
df2 = df[df.age > 3]
df2.createOrReplaceTempView("people")

In [ ]:
df3 = spark.sql("select * from people")
sorted(df2.collect()) == sorted(df3.collect())

In [ ]:
spark.catalog.dropTempView("people")

In [ ]:
# createTempView(self, name: str) -> None
#  |      Creates a local temporary view with this :class:`DataFrame`.
#  |      
#  |      The lifetime of this temporary table is tied to the :class:`SparkSession`
#  |      that was used to create this :class:`DataFrame`.
#  |      throws :class:`TempTableAlreadyExistsException`, if the view name already exists in the
#  |      catalog.

In [ ]:
# crossJoin(self, other: 'DataFrame') -> 'DataFrame'
#  |      Returns the cartesian product with another :class:`DataFrame`.

from pyspark.sql import Row
df = spark.createDataFrame([(14, "Tom"), (23, "Alice"), (16, "Bob")], ["age", "name"])
df2 = spark.createDataFrame([Row(name="Tom", height=198), Row(name="Bob", height=175)])

# df.crossJoin(df2.name).show()
df.crossJoin(df2).show()

# Number of output rows
# rows(df) × rows(df2) = 3 × 2 = 6 rows

# Final Output:
# +---+-----+-----+------+
# |age| name| name|height|
# +---+-----+-----+------+
# |14 | Tom | Tom | 198  |
# |14 | Tom | Bob | 175  |
# |23 |Alice| Tom | 198  |
# |23 |Alice| Bob | 175  |
# |16 | Bob | Tom | 198  |
# |16 | Bob | Bob | 175  |
# +---+-----+-----+------+

# Important warning ⚠️
# crossJoin() is very expensive for large datasets:
# 1. No join condition
# 2. Massive data explosion (m X n rows assuming m and n are the number of rows in the two DataFrames)
# 3. Can crash jobs if used unintentionally

In [ ]:
# withColumn - very useful function in real life. Allows you to add a new column, update an existing col value, change the datatype of a col

df = spark.createDataFrame(data = [("Shishir", 25, "Male"), ("Rahul", 27, "Male")], schema = ["Name", "Age", "Gender"])
df.show()
df.printSchema()
# By default, spark treats a number as long. Say, we need to change the datatype of age to int

In [ ]:
# withColumn(colName: str, col: pyspark.sql.column.Column) -> 'DataFrame' 

# Returns a new :class:`DataFrame` by adding a column or replacing the
#     existing column that has the same name.

# The column expression must be an expression over this :class:`DataFrame`; attempting to add
#     a column from some other :class:`DataFrame` will raise an error.

from pyspark.sql.functions import col

# colName is caseInsesitive - meaning if the col exists (doesn't matter if the name is in uppercase or lowercase, spark is going to modify the existing column)
# Original df had 'Age' column, but within withColumn function we passed 'age'. Spark identifies and knows that we want to modify the exisitng 'Age' column. 
# df.withColumn(colName, col)-- 2nd parameter is a Column instance always
df2 = df.withColumn('age', col('age').cast('Integer'))
df2.printSchema()
df2.show()

In [ ]:
# col(col: str) -> pyspark.sql.column.Column
#     Returns a :class:`~pyspark.sql.Column` based on the given column name.

# lit(col: Any) -> pyspark.sql.column.Column
#     Creates a :class:`~pyspark.sql.Column` of literal value.

# If you want to provide hard-coded value as col instance, use lit() function
            
from pyspark.sql.functions import lit
    
# Introduce a new column -- country col doesn't exist. 
df3 = df2.withColumn('country', lit("India"))
df3.show()

In [ ]:
# introduce a new column, using values from another col in df
df3.withColumn('doubleAge', df2.age * 2).show()
df3.withColumn('doubleAge', col("age") * 2).show()

# There are 2 ways to pick a column 
# 1. df.colName  --> returns column instance
# 2. using col() function -- col(colName) --> returns column instance

In [ ]:
df3.age

In [ ]:
col('age')

In [ ]:
# Imp NOTE: Pyspark DataFrames are immutable. Whatever transformation you apply, it's going to return you a new DataFrame. 
# Changes won't be performed in the existing dataframe

In [ ]:
# withColumnRenamed(existing: str, new: str) -> 'DataFrame' method of pyspark.sql.dataframe.DataFrame instance
#     Returns a new :class:`DataFrame` by renaming an existing column.
#     This is a no-op if the schema doesn't contain the given column name.

# df3.withColumn(colName: str, col: pyspark.sql.column.Column) -- was used to change/maniupulate column values or to introduce new column
# df3.withColumnRenamed(oldColName: str, newColName: str) -- used for renaming an existing column. If the column doesn't exist, no operation is performed

df4 = df3.withColumnRenamed('age', 'age2')

df3.show()

df4.show()

#NOTE: It's clearly evident that the transformation didn't change the existing df. Instead it returned a new dataFrame with the updated colName.

help(df3.withColumnRenamed)

In [ ]:
# class StructType(DataType)
# Struct type, consisting of a list of :class:`StructField`.
#  |  
#  |  This is the data type representing a :class:`Row`.
#  |  
#  |  Iterating a :class:`StructType` will iterate over its :class:`StructField`\s.
#  |  A contained :class:`StructField` can be accessed by its name or position.
        
from pyspark.sql.types import *

struct1 = StructType([StructField("f1", StringType())])

struct2 = StructType().add(StructField("f1", StringType()))

struct1 == struct2

In [ ]:
data = [("Alice", ["Java", "Scala"]), ("Bob", ["Java", "Scala"])]

schema = StructType([\
               StructField("name", StringType()), \
               StructField("languageSkills", ArrayType(StringType()))
           ])
                     
df = spark.createDataFrame(data = data, schema = schema)
df.show()
df.printSchema()

In [ ]:
# StructType, StructField, IntegerType, StringType, ArrayType etc are different classes in pyspark.sql.types module
# StructType class has an add() method which accepts a StructField or colName:str, DataType, Nullability:bool.
# These are just different types that pyspark supports for creating StructType. All are conceptually same.
# Interesting thing to NOTE: DataType can be another StructType as well. This allows us to define complex structure types.

data = [(1, ("Shishir", "Singh"), 25), (2, ("Rahul", "Patil"), 27)]


structName = StructType([StructField("firstName", StringType()), \
                         StructField("lastName", StringType()),])

schema = StructType([\
               StructField("id", IntegerType()), \
               StructField("name", structName), \
               StructField("age", IntegerType())
           ])

schema2 = StructType().add(StructField("id", IntegerType())) \
                      .add(StructField("name", structName)) \
                      .add(StructField("age", IntegerType()))

schema3 = StructType().add("id", IntegerType()) \
                      .add("name", structName) \
                      .add("age", IntegerType())

df = spark.createDataFrame(data = data, schema = schema3)

print(schema == schema2 == schema3)

df.show()

display(df)

df.printSchema()


In [ ]:
# class ArrayType in module pyspark.sql.types:
# ArrayType(elementType: pyspark.sql.types.DataType, containsNull: bool = True)
# The array can contain null (None) values by default

from pyspark.sql.types import * 

# AssertionError: elementType <class 'pyspark.sql.types.StringType'> should be an instance 
#     of <class 'pyspark.sql.types.DataType'>

# ArrayType(StringType) -- x elementType should be an instance of DataType. 
# That is why it's important to add () to StringType() bcoz we need to pass instance of DataType()

ArrayType(StringType()) # by default, array can null 

In [ ]:
ArrayType(StringType(), False) == ArrayType(StringType())

In [ ]:
data = [("abc", [1,2]), ("def", [3,4])]
df = spark.createDataFrame(data, schema = ["id", "numbers"])
df.show()

# spark automatically inferred the datatype by scanning the data. Saw array containing numbers. 
# By default, spark stores numbers in long.
df.printSchema()

# explicility defining schema
schema = StructType().add(StructField("id", StringType())) \
                     .add(StructField("numbers", ArrayType(IntegerType())))

df = spark.createDataFrame(data, schema = schema)
df.show()

# Now we have defined our schema - and array contains integer's
df.printSchema()


In [ ]:
data = [(1,2), (3,4)]

# passes schema as DDL formatted string
df = spark.createDataFrame(data = data, schema = "num1 int, num2 int")
df.show()
df.printSchema()

# say we want to add a new column
# syntax: withColName(colName: str, col: pyspark.sql.column.Column)
# array() -> pyspark.sql.column.Column
# Parameters
#     ----------
#     cols : :class:`~pyspark.sql.Column` or str
#         column names or :class:`~pyspark.sql.Column`\s that have
#         the same data type.

from pyspark.sql.functions import array, col

# you can access the array values using positions
df.withColumn('numbers', array(col("num1"), col("num2"))) \
  .withColumn("firstVal", col("numbers")[0]).show()

# array() is a column expression function that:
# Combines multiple column values into a single ArrayType column

# Function signature (what Spark expects)

# array(cols: Column or str) → Column

# Key points:
# 1. Accepts one or more columns
# 2. All columns must be same data type
# 3. Returns a Column of ArrayType

In [ ]:
from pyspark.sql.functions import *
help(array)

In [ ]:
# some commonly used functions with array in pyspark

# explode(col: 'ColumnOrName') -> pyspark.sql.column.Column
#     Returns a new row for each element in the given array or map.
#     Uses the default column name `col` for elements in the array and
#     `key` and `value` for elements in the map unless specified otherwise.

#  Parameters
#     ----------
#     col : :class:`~pyspark.sql.Column` or str
#         target column to work on.
    
    
#     Returns
#     -------
#     :class:`~pyspark.sql.Column`
#         one row per array item or map key value.


# Key characteristics of explode()
# Property	    Behavior
# Input	        Array or Map
# Output rows	    One per element
# Row count	    Increases
# Other columns	Duplicated
# Lazy	        Yes

from pyspark.sql.functions import explode, col

df = spark.createDataFrame([Row(a=1, intlist=[1,2,3], mapfield={"a": "b", "c": "d"})])
df.show()

# added a new column
df.withColumn('num', explode(col('intlist'))).show()

# +---+--------+----------------+---+
# | a | intlist| mapfield       |num|
# +---+--------+----------------+---+
# | 1 | [1,2,3]| {a -> b, c -> d}| 1 |
# | 1 | [1,2,3]| {a -> b, c -> d}| 2 |
# | 1 | [1,2,3]| {a -> b, c -> d}| 3 |
# +---+--------+----------------+---+

# selecting just the explode column
df.select(explode(col('intlist'))).show()

df.select(explode(col('intlist')).alias('intNum')).show()

# NOTE: calling explode() on map field produced 2 columns - key and value. Spark automatically creates these columns for us when we call explode() on a map field.
# Each map entry produces a new row with key and value columns.
df.select(explode(col('mapfield'))).show()
# +---+-----+
# |key|value|
# +---+-----+
# | a | b   |
# | c | d   |
# +---+-----+
# For maps: Spark produces key and value columns automatically

# Important caution ⚠️

# explode() can:
# 1. Multiply rows significantly
# 2. Increase memory and shuffle costs

# Always be careful with large arrays.

In [ ]:
# explode() vs flatMap()

...

In [ ]:
# split(str: 'ColumnOrName', pattern: str, limit: int = -1) -> pyspark.sql.column.Column
#     Splits str around matches of the given pattern.
    
#     Parameters
#     ----------
#     str : :class:`~pyspark.sql.Column` or str
#         a string expression to split
#     pattern : str
#         a string representing a regular expression. The regex string should be
#         a Java regular expression.
#     limit : int, optional
#         an integer which controls the number of times `pattern` is applied.
    
#         * ``limit > 0``: The resulting array's length will not be more than `limit`, and the
#                          resulting array's last entry will contain all input beyond the last
#                          matched pattern.
#         * ``limit <= 0``: `pattern` will be applied as many times as possible, and the resulting
#                           array can be of any size.
    
#         .. versionchanged:: 3.0
#            `split` now takes an optional `limit` field. If not provided, default limit value is -1.
    
    
# split(ColumnOrName) -- splits the string around the delimiter and returns an array
data = [(1, "Shishir", ".net, java, aws"), (1, "Rahul", "c++, python, golang")]
df = spark.createDataFrame(data = data, schema = ["id", "name", "skills"])
df.show()
df.printSchema()


from pyspark.sql.functions import split, col

df2 = df.withColumn('skillsArray', split(col('skills'), ','))
df2.show()
df2.printSchema()

In [ ]:
# array(*cols: Union[ForwardRef('ColumnOrName'), List[ForwardRef('ColumnOrName_')], Tuple[ForwardRef('ColumnOrName_'), ...]]) -> pyspark.sql.column.Column
#     Creates a new array column.

#  Parameters
#     ----------
#     cols : :class:`~pyspark.sql.Column` or str
#         column names or :class:`~pyspark.sql.Column`\s that have
#         the same data type.
    
#     Returns
#     -------
#     :class:`~pyspark.sql.Column`
#         a column of array type.

df = spark.createDataFrame([("Alice", 2), ("Bob", 5)], ("name", "age"))
df.show()

df.select(array(col('age'), col('age')).alias('numbers')).show()

df.select(array('age', 'age').alias('numbers')).show()

df = spark.createDataFrame(data = [("Alice", "java", "aws"), ("Bob", "python", 'gcp')], 
                        schema = ["name", "primarySkill", "secondarySkill"])

df.show()

df.withColumn('skills', array(col('primarySkill'), col('secondarySkill'))).show()

In [ ]:
df = spark.createDataFrame(data = [("Alice", "java", "aws"), ("Bob", "python", 'gcp')], 
                        schema = ["name", "primarySkill", "secondarySkill"])

df.show()

df2 = df.withColumn('skills', array(col('primarySkill'), col('secondarySkill')))

df2.show()

# this is case sensitive. 

from pyspark.sql.functions import array_contains

# notice: how the output changes when I change the casing of 'java' string
print('Does it contain JAVA?')
df2.withColumn('hasJavaSkill', array_contains(col('skills'), 'JAVA')).show()

print('Does it contain Java?')
df2.withColumn('hasJavaSkill', array_contains(col('skills'), 'Java')).show()

print('Does it contain java?')
df2.withColumn('hasJavaSkill', array_contains(col('skills'), 'java')).show()

In [ ]:
# Help on class MapType in module pyspark.sql.types:

# class MapType(DataType)
#  |  MapType(keyType: pyspark.sql.types.DataType, valueType: pyspark.sql.types.DataType, valueContainsNull: bool = True)
#  |  
#  |  Map data type.
#  |  
#  |  Parameters
#  |  ----------
#  |  keyType : :class:`DataType`
#  |      :class:`DataType` of the keys in the map.
#  |  valueType : :class:`DataType`
#  |      :class:`DataType` of the values in the map.
#  |  valueContainsNull : bool, optional
#  |      indicates whether values can contain null (None) values.
#  |  
#  |  Notes
#  |  -----
#  |  Keys in a map data type are not allowed to be null (None).

from pyspark.sql.types import *
from pyspark.sql.functions import *

data = [("Shishir", {"eye": "black", "height": 180}), ("Rahul", {"eye": "brown", "height": 175})]

# we have not specified the schema explicitly. Spark infers the schema & Sees properties is of map type
# where key and value are both string. Although we passed height as int, it treats it as string
df = spark.createDataFrame(data = data, schema = ["name", "properties"])
df.show()
df.printSchema()

# defining schema explicitly -- name of string and properties of map type

# MapType(keyType: pyspark.sql.types.DataType, valueType: pyspark.sql.types.DataType, valueContainsNull: bool = True)
# Parameters
#  |  ----------
#  |  keyType : :class:`DataType`
#  |      :class:`DataType` of the keys in the map.
#  |  valueType : :class:`DataType`
#  |      :class:`DataType` of the values in the map.
#  |  valueContainsNull : bool, optional
#  |      indicates whether values can contain null (None) values.
#  |  
#  |  Notes
#  |  -----
#  |  Keys in a map data type are not allowed to be null (None).

schema = StructType().add(StructField("name", StringType())) \
                     .add(StructField("properties", MapType(StringType(), StringType())))

df = spark.createDataFrame(data = data, schema = schema)
df.show(truncate=False)
df.printSchema()


# accessing map's key values

# df.properties or col('properties') both will return map column. To access a key's value - use mp[keyName]
# col('properties') is the map column which I want to access and then the key
df = df.withColumn('eyeColor', col('properties')['eye']) \
  .withColumn('height', col('properties')['height']) 

df.show(truncate=False)

In [ ]:
# explode(col: 'ColumnOrName') -> pyspark.sql.column.Column

# explode() on map column basically produces 2 columns for key and value. Works the same as in case of array, just that in arrays it produces 1 column
df.select(df.name, df.properties, df.eyeColor, df.height, explode(col("properties")).alias("key", "value")) \
.show(truncate=False)

In [ ]:
df.show(truncate=False)

# map_keys(col: 'ColumnOrName') -> pyspark.sql.column.Column
#     Collection function: Returns an unordered array containing the keys of the map.


from pyspark.sql.functions import map_keys, map_values
# added 2 new columns extracting keys and values using map_keys() and map_values() functions.
# map_keys() and map_values() both return pyspark.sql.column.Column instance of array type
df_new = df.withColumn('mapKeys', map_keys(col('properties'))) \
 .withColumn('mapValues', map_values(col('properties'))) 

df_new.show()
df_new.printSchema()


In [ ]:
# Below line imports all public names (classes, functions, constants) defined in the pyspark.sql.types module 
# into our current namespace
from pyspark.sql.types import *

# this is how you create a complex schema
schema = StructType().add(StructField("name", StringType())) \
            .add(StructField("properties", MapType(StringType(), StringType()))) \
            .add(StructField("eyeColor", StringType())) \
            .add(StructField("height", StringType())) \
            .add(StructField("mapKeys", ArrayType(StringType()))) \
            .add(StructField("mapValues", ArrayType(StringType()))) 

# pyspark is smart enough & perfectly capable to determine that it's a StructField.
# same schema defined without using StructField() -- by just specifying the colName, DataType(), nullability[optional]
schema = StructType().add("name", StringType()) \
            .add("properties", MapType(StringType(), StringType())) \
            .add("eyeColor", StringType()) \
            .add("height", StringType()) \
            .add("mapKeys", ArrayType(StringType())) \
            .add("mapValues", ArrayType(StringType()))
                 
schema
                 

In [ ]:
# Commonly used Pyspark Modules

# Modules                   Purpose                     Example
# pyspark.sql           	Core SQL/DataFrame API     	SparkSession, DataFrame
# pyspark.sql.functions	    Built-in column functions 	col, when, lit, sum, avg, corr
# pyspark.sql.types	        Data types & schemas      	StringType, StructType
# pyspark.sql.window    	Window functions         	Window.partitionBy
# pyspark.ml                Machine learning            Pipeline, VectorAssembler



# Hierarchy is something like this

# Package
#  └── Module
#       ├── Classes        → define objects with state + behavior
#       ├── Functions      → standalone reusable logic
#       ├── Constants      → fixed values / configuration
#       ├── Variables      → runtime data
#       └── Imports        → things pulled from other modules

        
# Example: pyspark.sql.functions (a module)
# pyspark.sql.functions   ← module
#  ├── col()              ← function
#  ├── avg()              ← function
#  ├── when()             ← function
#  ├── lit()              ← function
#  └── ...
    
    
# pyspark.sql.types       ← module
#  ├── StringType         ← class
#  ├── IntegerType        ← class
#  ├── StructType         ← class
#  ├── StructField        ← class
#  └── ...

# Key clarifications

# 1. A module is just a .py file (or compiled equivalent)
# 2. A package is a folder of modules
# 3. Inside a module you can have:
#     Classes
#     Functions
#     Constants
#     Variables
#     Imports
# 4. Functions do not have to be inside classes in Python
# 5. Classes are for grouping data + behavior; functions are for standalone behavior



# How come a function exists separately inside a module. Generally functions are defined in a class right?
# You’re right that in many OOP designs, functions live inside classes.
# But Python supports multiple styles:

# Object-Oriented

# Functional

# Procedural

# PySpark mixes these styles for usability.

# Example:
    
# from pyspark.sql import functions as F

# df.select(F.col("age"), F.avg("salary"))


# Here: col, avg are just functions, not class methods.
# They return a Column object.
# That Column is later used by DataFrame methods like select, agg.

# So the design is:
# function → returns an object → consumed by class method

# Why Spark chose this design

# 1. SQL-like feel
# select(col("a"), sum("b"))

# Looks similar to SQL:
# SELECT a, SUM(b) FROM table

# 2. Language-agnostic API design

# Spark is written in Scala. Python API mirrors Scala API:

# col("age")
# avg("salary")

In [ ]:
lst = [1,2,3,4]
print(lst)

In [ ]:
from pyspark.sql.types import Row

help(Row)